In [5]:
import json
from typing import Literal

from openai import OpenAI
from pydantic import BaseModel, Field

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex
import search_tools


class Agent:

    FAKE_TOOL_NAME = "structure_result"

    search_tool = {
        "type": "function",
        "name": "search",
        "description": "Search the documentation database for relevant results based on a query string.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to look up in the index",
                }
            },
            "required": ["query"],
        },
    }

    add_entry_tool = {
        "type": "function",
        "name": "add_entry",
        "description": "Add a new documentation entry to the index.",
        "parameters": {
            "type": "object",
            "properties": {
                "filename": {
                    "type": "string",
                    "description": "The source filename associated with the entry",
                },
                "title": {
                    "type": "string",
                    "description": "The title of the documentation entry",
                },
                "description": {
                    "type": "string",
                    "description": "A short description summarizing the entry",
                },
                "content": {
                    "type": "string",
                    "description": "The full content of the documentation entry",
                },
            },
            "required": ["filename", "title", "description", "content"],
        },
    }

    def __init__(
        self,
        llm_client,
        model,
        instructions,
        search_index_tools,
        tools=None,
        output_type=None,
    ):
        self.llm_client = llm_client
        self.model = model
        self.instructions = instructions
        self.search_index_tools = search_index_tools
        self.output_type = output_type
        self.tools = [self.search_tool, self.add_entry_tool]

        if tools is not None:
            self.tools.extend(tools)

        if self.output_type is not None:
            self.tools.append(
                self.create_fake_tool(
                    output_type=self.output_type,
                    name=self.FAKE_TOOL_NAME,
                )
            )

    def create_fake_tool(self, output_type, name="structure_result"):
        schema = output_type.model_json_schema()
        schema["type"] = "object"
        schema["additionalProperties"] = False

        return {
            "type": "function",
            "name": name,
            "description": "Call this function when you're ready to show the final result as structured data.",
            "strict": True,
            "parameters": schema,
        }

    def make_call(self, tool_call):
        arguments = json.loads(tool_call.arguments)
        name = tool_call.name

        if name == "search":
            result = self.search_index_tools.search(**arguments)
        elif name == "add_entry":
            result = self.search_index_tools.add_entry(**arguments)
        else:
            raise ValueError(f'Unknown executable tool "{name}"')

        return {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": json.dumps(result),
        }

    def loop(self, user_prompt, message_history=None):
        if message_history is None:
            message_history = [
                {"role": "system", "content": self.instructions},
            ]

        message_history.append({"role": "user", "content": user_prompt})

        iteration_number = 0

        while True:
            response = self.llm_client.responses.create(
                model=self.model,
                input=message_history,
                tools=self.tools,
            )

            print(f"iteration number {iteration_number}...")

            has_function_calls = False
            structured_output = None

            for message in response.output:
                if message.type == "function_call":
                    print(f"executing {message.name}({message.arguments})...")

                    if message.name == self.FAKE_TOOL_NAME:
                        output_json = json.loads(message.arguments)
                        structured_output = self.output_type.model_validate(output_json)
                        continue

                    message_history.append(message)
                    tool_call_output = self.make_call(message)
                    message_history.append(tool_call_output)
                    has_function_calls = True

                elif message.type == "message":
                    message_history.append(message)
                    text = message.content[0].text
                    print("ASSISTANT:", text)

            iteration_number += 1
            print()

            if structured_output is not None:
                message_history.append(
                    {
                        "role": "assistant",
                        "content": structured_output.answer,
                    }
                )
                return message_history, structured_output

            if not has_function_calls:
                return message_history, None

    def run_single_turn(self, user_prompt):
        _, structured_output = self.loop(
            user_prompt=user_prompt,
            message_history=None,
        )
        return structured_output

    def qna(self):
        message_history = [
            {"role": "system", "content": self.instructions},
        ]

        while True:
            user_prompt = input("You: ").strip()

            if user_prompt.lower() in {"stop", "exit", "quit"}:
                break

            if not user_prompt:
                continue

            message_history, structured_output = self.loop(
                user_prompt=user_prompt,
                message_history=message_history,
            )

            if structured_output is not None:
                print("STRUCTURED OUTPUT:", structured_output)

In [6]:
import search_tools

instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

openai_client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)

index.fit(chunked_docs)

search_tools_object = search_tools.SearchIndexTools(index)
result = search_tools_object.search("dashboard")
result

[{'start': 9000,
  'content': 'ined custom parameters\n\nIn these cases, use `metric_labels` to specify what exactly you want to plot.\n\n**Example**. To plot the share of categories inside "Denials" column:\n\n```python\nproject.dashboard.add_panel(\n             DashboardPanelPlot(\n                title="Denials",\n                subtitle = "Number of denials.",\n                size="half",\n                values=[\n                    PanelMetric(\n                        legend="""{{label}}""",\n                        metric="UniqueValueCount", # <- metric from TextEvals Preset that computes distinct values\n                        metric_labels={"column": "denials", #column name\n                                       "value_type": "share" #metric result\n                                       } \n                    ),\n                ],\n                plot_params={"plot_type": "bar", "is_stacked": True},\n            ),\n            tab="My tab",\n        )\n```\n\n**Col

In [ ]:
# before split
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""


class AgentResponse(BaseModel):
    """
    Structured final response from the agent.
    """

    answer: str = Field(
        description="The main answer to the user's question in markdown"
    )
    found_answer: bool = Field(
        description="True if relevant information was found in the documentation"
    )
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Confidence score from 0.0 to 1.0 indicating how certain the answer is",
    )
    confidence_explanation: str = Field(
        description="Explanation about the confidence level"
    )
    answer_type: Literal[
        "how-to",
        "explanation",
        "troubleshooting",
        "comparison",
        "reference",
    ] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(
        description="Suggested follow-up questions the user might want to ask"
    )


openai_client = OpenAI()

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)

files = reader.read()
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)

index.fit(chunked_docs)

agent = Agent(
    llm_client=openai_client,
    model="gpt-4.1-mini",
    
    instructions=instructions,
    index=index,
    output_type=AgentResponse,
)



TypeError: Agent.__init__() got an unexpected keyword argument 'index'

In [9]:
# One prompt, no CLI loop
output = agent.run_single_turn("Give me an overview of dashboard panels")
print(output.answer)

iteration number 0...
executing search({"query":"dashboard panels overview"})...

iteration number 1...
ASSISTANT: In the first search, I queried "dashboard panels overview" because the user requested an overview of dashboard panels, which should capture general documentation explaining what dashboard panels are and how they are used. The search results show documentation about dashboard panels, including adding tabs and panels, types of panels, filtering metrics, and configuring panels via UI and API. However, this seems to focus on detailed instructions rather than a brief overview, so further searches might help capture a high-level concise overview or different perspectives such as types and key features.

Next, I will do two more searches: 
1) "dashboard panel types" - to find documentation specifying the different types of panels available to understand the key ways users can visualize data.
2) "dashboard panel features" - to find documentation highlighting important features or 

In [ ]:
# Interactive CLI session

agent.qna()